# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'portfolio project',
   'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'portfolio project', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'skills page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 5 relevant links


{'links': [{'type': 'company homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'linkedin page', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [11]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


{'links': [{'type': 'home page', 'url': 'https://huggingface.co/'},
  {'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'company page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'community page', 'url': 'https://discuss.huggingface.co'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'Zhihu page', 'url': 'https://www.zhihu.com/org/huggingface'},
  {'type': 'Endpoints product page',
   'url': 'https://endpoints.huggingface.co'},
  {'type': 'Status page', 'url': 'https://status.huggingface.co/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [18]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [13]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
NEW
Google Gemma 4 is here 💫
Storage Buckets: AI-native object storage
GGML and llama.cpp join Hugging Face 🔥
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled
Updated
10 days ago
•
487k
•
2.17k
CohereLabs/cohere-transcribe-03-2026
Updated
about 19 hours ago
•
84.6k
•
750
google/gemma-4-31B-it
Updated
1 day ago
•
76.2k
•
537
baidu/Qianfan-OCR
Updated
8 days ago
•
27k
•
836
mistralai/Voxtral-4B-TTS-2603
Updated
3 days ago
•
4.76k
•
641
Browse 2M+ models
Spaces
Running
on
Zero
MCP
1.78k
Wan2.2 14B Preview
🐌
1.78k
generate a video from an image with a text prompt
Running
on
Zero
MCP
Featured
620
FireRed Image Edit 1.0 Fast


In [20]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [21]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [22]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 9 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nNEW\nGoogle Gemma 4 is here 💫\nStorage Buckets: AI-native object storage\nGGML and llama.cpp join Hugging Face 🔥\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nJackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled\nUpdated\n10 days ago\n•\n487k\n•\n2.17k\nCohereLabs/cohere-transcribe-03-2026\nUpdated\nabout 19 hours ago\n•\n84.6k\n•\n750\ngoogle/gemma-4-31B-it\nUpdated\n1 day ago\n•\n76.2k\n•\n538\nbaidu/Qianfan-OCR\nUpdated\n8 d

In [23]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [24]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 14 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the AI community building the future. It serves as the leading open collaboration platform for the global machine learning community — empowering engineers, researchers, and end users to create, discover, and innovate together. Hugging Face hosts one of the largest open repositories of machine learning models, datasets, and applications, fostering transparency, openness, and ethical AI development.

---

## What Hugging Face Offers

- **Models Library:**  
  Access over 2 million pre-trained ML models across various modalities including text, images, speech, and more. Popular models include Google Gemma 4, Qwen series, CohereLabs transcribers, and many others.

- **Datasets Hub:**  
  Explore and share from a collection of 500,000+ datasets that fuel ML research and applications across domains.

- **Spaces:**  
  Run and discover AI-powered applications built by the community, such as video generation from images or advanced image editing.

- **Buckets:**  
  AI-native object storage designed for efficient machine learning workflows.

- **Open Source Stack:**  
  Hugging Face provides a complete open-source ecosystem to accelerate ML development including APIs, tools, and frameworks that support the entire ML lifecycle.

---

## Community & Culture

Hugging Face fosters a passionate, collaborative community dedicated to responsible and ethical AI. It encourages open exchange of ideas and collective problem-solving. The inclusive atmosphere supports learning and sharing with thousands of contributors worldwide.

- Accessibility for all skill levels from beginners to experts
- Transparency and ethical standards in AI development
- Growth-driven with continuous updates to tools and models
- Supportive environment emphasizing knowledge exchange

---

## Customers & Users

Hugging Face is trusted by a diverse audience:

- Individual ML engineers and data scientists searching for cutting-edge models and datasets
- Academic researchers driving innovation in AI
- Enterprises leveraging AI models tailored to business needs
- Developers building production-grade AI applications rapidly

The platform powers a wide range of use cases from natural language processing, computer vision, speech recognition to generative AI.

---

## Careers & Opportunities

Join Hugging Face and be part of the AI revolution. The company offers exciting career opportunities for:

- Machine Learning Researchers and Engineers
- Open Source Developers and Contributors
- Product Managers and Designers passionate about AI
- Community and Developer Relations Specialists

Hugging Face values creativity, collaboration, and a strong commitment to building open AI. Employees enjoy working at the forefront of technology with a global impact.

---

## Why Choose Hugging Face?

- **Leading Open ML Platform:** Millions of models & datasets at your fingertips
- **AI-Native Storage & Tools:** Optimized infrastructure for ML needs
- **Thriving Global Community:** Exchange knowledge and collaborate worldwide
- **Ethical & Open AI Future:** Committed to transparency and responsible AI

Explore the future of machine learning with Hugging Face — where innovation meets community.

---

**Learn more and get involved:**  
[https://huggingface.co](https://huggingface.co)  

Join the AI community building the future today!

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [25]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [26]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


# Hugging Face Brochure

---

## Who We Are

**Hugging Face** is the AI community building the future. As the central collaboration platform for the machine learning (ML) community, we empower millions of people worldwide to create, discover, and share open-source machine learning models, datasets, and applications. Our fast-growing community and talented science team are at the forefront of the AI revolution, fostering an open, ethical, and innovative AI ecosystem.

---

## What We Offer

- **Hugging Face Hub:**  
  A centralized place where anyone—researchers, engineers, scientists, and developers—can share, explore, and experiment with over **2 million models**, **500k+ datasets**, and **1 million+ AI applications** across modalities like text, images, and more.

- **Spaces:**  
  Deploy and run ML applications in an interactive web environment directly on the platform.

- **Storage Buckets:**  
  AI-native object storage built for machine learning workflows.

- **Enterprise & Team Plans:**  
  Scaling your organization with advanced AI platform tools including:
  - Enterprise-grade security with Single Sign-On (SSO)
  - Granular access controls & resource groups
  - Audit logs and analytics dashboard
  - Advanced compute options and ZeroGPU quota boosts for performance
  - Private datasets viewer and storage expansion
  - Centralized token and billing management

---

## Our Community

We host a vibrant and inclusive global community where collaboration and knowledge sharing drive rapid AI innovation. Our platform enables:

- Machine learning engineers and scientists to collaborate across disciplines.
- Developers and end-users to experiment with state-of-the-art models.
- Open-source contributors to share ethical AI work and accelerate progress.
  
Trending projects often include cutting-edge language models, image generation, transcription tools, and innovative reasoning models contributed and updated continuously by the community.

---

## Company Culture

At Hugging Face, we value:

- **Open Collaboration:** Encouraging sharing and teamwork to push AI boundaries together.
- **Ethical AI Development:** Committed to building an open and fair AI future.
- **Innovation and Excellence:** Supporting a science-driven approach with cutting-edge research.
- **Diversity & Inclusion:** Empowering the next generation of AI talent from diverse backgrounds.

---

## Careers

Join Hugging Face if you're passionate about transforming AI for the better. We look for curious, talented individuals eager to work at the intersection of open-source, machine learning, and community innovation. 

Explore opportunities to be part of our team advancing ML research, engineering scalable AI systems, or supporting our extensive global community.

Visit our [Careers page](https://huggingface.co/careers) to apply.

---

## Connect With Us

- Website: [huggingface.co](https://huggingface.co)  
- GitHub: [github.com/huggingface](https://github.com/huggingface)  
- Twitter: [@huggingface](https://twitter.com/huggingface)  
- LinkedIn: [Hugging Face](https://linkedin.com/company/huggingface)  
- Discord: Join our community chat for support and collaboration.

---

## Brand Highlights

- Vibrant brand colors: Yellow (#FFD21E), Orange (#FF9D00), and Charcoal Gray (#6B7280).
- Logo and identity that reflect our friendly, forward-thinking mission.
  
---

**Hugging Face — Together, shaping the future of AI.**

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>